# Modulo 8 · Lezione 9 — Addestramento e Monitoraggio

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | TensorBoard: visualizzazione dell'addestramento |
| 2 | Learning Rate Schedule |
| 3 | Callback personalizzati |
| 4 | Diagnosi avanzata dei modelli |
| 5 | Flusso di lavoro professionale |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os, datetime

tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow: {tf.__version__}")

# Dataset base per gli esempi
(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.mnist.load_data()
X_tr = X_tr_raw.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te_raw.reshape(-1, 784).astype('float32') / 255.0
print(f"MNIST caricato: {X_tr.shape}")

---
## 1. TensorBoard: visualizzazione dell'addestramento

```
TensorBoard è lo strumento di visualizzazione di TensorFlow.
Permette di vedere in tempo reale (o dopo l'addestramento):
  - Curve di loss e metriche
  - Istogrammi di pesi e gradienti per strato
  - Grafi del modello
  - Immagini e audio di esempio
  - Proiezione degli embedding in 3D
  - Profiling delle prestazioni

Uso:
  1. Aggiungere il callback TensorBoard all'addestramento
  2. Lanciare: tensorboard --logdir logs/
  3. Aprire http://localhost:6006 nel browser

In Jupyter:
  %load_ext tensorboard
  %tensorboard --logdir logs/
```

In [ ]:
# Configurare TensorBoard
log_dir = 'logs/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
print(f"Log in: {log_dir}")

tf.random.set_seed(42)
modelo_tb = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu', name='capa1'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu', name='capa2'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax', name='salida'),
])
modelo_tb.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

tb_callback = keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,      # istogrammi dei pesi ogni N epoche
    write_graph=True,      # salvare il grafo del modello
    write_images=False,
    update_freq='epoch',   # frequenza di log: 'epoch' o 'batch'
    profile_batch=0,       # 0 = disattivare il profiling
)

hist_tb = modelo_tb.fit(
    X_tr, y_tr_raw,
    epochs=10, batch_size=256,
    validation_split=0.1,
    callbacks=[tb_callback],
    verbose=0
)

_, acc = modelo_tb.evaluate(X_te, y_te_raw, verbose=0)
print(f"Test accuracy: {acc:.4f}")
print(f"\nPer vedere TensorBoard, esegui nel terminale:")
print(f"  tensorboard --logdir {os.path.dirname(log_dir)}")
print("Poi apri http://localhost:6006")

In [ ]:
# Scrivere metriche personalizzate in TensorBoard
log_custom = 'logs/custom/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
writer = tf.summary.create_file_writer(log_custom)

# Simulare l'addestramento manuale e il log delle metriche
tf.random.set_seed(42)
m_manual = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
optimizer = keras.optimizers.Adam(0.001)
loss_fn   = keras.losses.SparseCategoricalCrossentropy()

dataset = tf.data.Dataset.from_tensor_slices((X_tr, y_tr_raw)).batch(256)

for epoch in range(5):
    losses = []
    for xb, yb in dataset:
        with tf.GradientTape() as tape:
            logits = m_manual(xb, training=True)
            loss   = loss_fn(yb, logits)
        grads = tape.gradient(loss, m_manual.trainable_variables)
        optimizer.apply_gradients(zip(grads, m_manual.trainable_variables))
        losses.append(float(loss))

    mean_loss = np.mean(losses)
    # Scrivere in TensorBoard
    with writer.as_default():
        tf.summary.scalar('train_loss_manual', mean_loss, step=epoch)
        tf.summary.scalar('lr', float(optimizer.learning_rate), step=epoch)
    print(f"Epoca {epoch+1}: loss={mean_loss:.4f}")

print(f"\nLog personalizzati in: {log_custom}")

---
## 2. Learning Rate Schedule

In [ ]:
print("""
Il tasso di apprendimento (lr) è l'iperparametro più importante.
Un lr fisso raramente è ottimale:

  Alto all'inizio → esplora lo spazio, scende rapidamente
  Basso alla fine → affina la convergenza

Strategie di lr schedule:

  1. Step Decay:           lr × fattore ogni N epoche
  2. Exponential Decay:    lr × e^(-k·t)
  3. Cosine Annealing:     lr oscilla come un coseno
  4. Warmup + Decay:       sale gradualmente, poi scende (Transformer)
  5. OneCycleLR:           lr_max a metà (molto efficace nella pratica)
  6. ReduceLROnPlateau:    già visto — riduce se val_loss si stagna
""")

In [ ]:
# Implementare e confrontare gli schedule
n_epocas = 50
epocas   = np.arange(n_epocas)

def step_decay(epoca, lr_inicial=0.01, factor=0.5, paso=10):
    return lr_inicial * (factor ** (epoca // paso))

def exp_decay(epoca, lr_inicial=0.01, k=0.05):
    return lr_inicial * np.exp(-k * epoca)

def cosine_annealing(epoca, lr_max=0.01, lr_min=1e-5, T=50):
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * epoca / T))

def warmup_decay(epoca, lr_max=0.01, warmup=10, T=50):
    if epoca < warmup:
        return lr_max * (epoca + 1) / warmup
    return lr_max * np.exp(-0.05 * (epoca - warmup))

schedules = {
    'Step Decay':        [step_decay(e) for e in epocas],
    'Exp Decay':         [exp_decay(e) for e in epocas],
    'Cosine Annealing':  [cosine_annealing(e) for e in epocas],
    'Warmup + Decay':    [warmup_decay(e) for e in epocas],
    'Costante (0.001)':  [0.001] * n_epocas,
}

fig, ax = plt.subplots(figsize=(12, 5))
for nombre, lr_vals in schedules.items():
    ax.semilogy(epocas, lr_vals, lw=2, label=nombre)
ax.set_xlabel('Epoca'); ax.set_ylabel('Learning rate (scala log)')
ax.set_title('Confronto tra Learning Rate Schedule')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

In [ ]:
# Applicare gli schedule in Keras
# Opzione 1: callback LearningRateScheduler
def schedule_fn(epoca):
    if epoca < 10:
        return 0.001 * (epoca + 1) / 10   # warmup
    return 0.001 * np.exp(-0.03 * (epoca - 10))

lr_callback = keras.callbacks.LearningRateScheduler(schedule_fn, verbose=0)

# Opzione 2: keras.optimizers.schedules
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.001,
    decay_steps=10000,
    alpha=1e-5  # lr minima
)

# Confrontare: lr fissa vs warmup+decay
resultados_lr = {}
for nombre, optimizer in [
    ('Adam lr=0.001 fissa', keras.optimizers.Adam(0.001)),
    ('CosineDecay',         keras.optimizers.Adam(lr_schedule)),
]:
    tf.random.set_seed(42)
    m = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64,  activation='relu'),
        layers.Dense(10,  activation='softmax'),
    ])
    m.compile(optimizer, 'sparse_categorical_crossentropy', ['accuracy'])
    h = m.fit(X_tr, y_tr_raw, epochs=15, batch_size=256,
              validation_split=0.1, verbose=0)
    _, acc = m.evaluate(X_te, y_te_raw, verbose=0)
    resultados_lr[nombre] = (h.history['val_accuracy'], acc)
    print(f"{nombre}: val_acc finale={h.history['val_accuracy'][-1]:.4f}, test_acc={acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
for nombre, (val_acc, _) in resultados_lr.items():
    ax.plot(val_acc, lw=2, label=nombre)
ax.set_xlabel('Epoca'); ax.set_ylabel('Val Accuracy')
ax.set_title('Effetto del LR Schedule')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 3. Callback personalizzati

In [ ]:
# I callback di Keras si implementano sovrascrivendo dei metodi

class MonitorAvanzado(keras.callbacks.Callback):
    """Callback che monitora, rileva problemi e salva il modello migliore."""

    def __init__(self, umbral_gap=0.05, ruta_guardado='mejor.keras'):
        super().__init__()
        self.umbral_gap    = umbral_gap
        self.ruta          = ruta_guardado
        self.mejor_val_acc = 0.0
        self.historia      = {'loss':[], 'val_loss':[], 'acc':[], 'val_acc':[], 'gap':[], 'lr':[]}

    def on_train_begin(self, logs=None):
        print(f"[Monitor] Addestramento iniziato. Gap massimo consentito: {self.umbral_gap}")

    def on_epoch_end(self, epoch, logs=None):
        acc     = logs.get('accuracy',     0)
        val_acc = logs.get('val_accuracy', 0)
        loss    = logs.get('loss',         0)
        val_loss= logs.get('val_loss',     0)
        gap     = acc - val_acc
        lr      = float(self.model.optimizer.learning_rate)

        for k, v in [('loss',loss),('val_loss',val_loss),('acc',acc),
                     ('val_acc',val_acc),('gap',gap),('lr',lr)]:
            self.historia[k].append(v)

        # Salvare se migliora
        if val_acc > self.mejor_val_acc:
            self.mejor_val_acc = val_acc
            self.model.save(self.ruta)
        # Allerta di overfitting
        if gap > self.umbral_gap and epoch > 5:
            print(f"  [Monitor] ALLERTA overfitting: gap={gap:.4f} (epoca {epoch+1})")
        # Log compatto ogni 5 epoche
        if (epoch + 1) % 5 == 0:
            print(f"  [Monitor] Epoca {epoch+1:3d}: loss={loss:.4f} val={val_loss:.4f} "
                  f"acc={acc:.4f} val_acc={val_acc:.4f} gap={gap:.4f}")

    def on_train_end(self, logs=None):
        print(f"[Monitor] Migliore val_accuracy: {self.mejor_val_acc:.4f}")
        self.plot()

    def plot(self):
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        axes[0].plot(self.historia['loss'],     'steelblue', label='Train')
        axes[0].plot(self.historia['val_loss'], 'coral',     label='Val')
        axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(self.historia['acc'],     'steelblue', label='Train')
        axes[1].plot(self.historia['val_acc'], 'coral',     label='Val')
        axes[1].fill_between(range(len(self.historia['gap'])),
                              self.historia['val_acc'],
                              self.historia['acc'], alpha=0.2, color='red', label='Gap')
        axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

        axes[2].semilogy(self.historia['lr'], 'purple', lw=2)
        axes[2].set_title('Learning rate'); axes[2].grid(True, alpha=0.3, which='both')

        for ax in axes:
            ax.set_xlabel('Epoca')
        plt.tight_layout(); plt.show()


# Usare il callback
tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

monitor = MonitorAvanzado(umbral_gap=0.03)
m.fit(X_tr, y_tr_raw, epochs=20, batch_size=256,
      validation_split=0.1, callbacks=[monitor], verbose=0)

---
## 4. Diagnosi avanzata dei modelli

In [ ]:
# Monitorare il flusso dei gradienti per strato
# Problema: gradienti molto piccoli → lo strato non impara
#           gradienti molto grandi → instabilità

tf.random.set_seed(42)
m_diag = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512, activation='relu', name='d1'),
    layers.Dense(256, activation='relu', name='d2'),
    layers.Dense(128, activation='relu', name='d3'),
    layers.Dense(64,  activation='relu', name='d4'),
    layers.Dense(10,  activation='softmax', name='salida'),
])

optimizer = keras.optimizers.Adam(0.001)
loss_fn   = keras.losses.SparseCategoricalCrossentropy()

X_batch = tf.constant(X_tr[:256])
y_batch = tf.constant(y_tr_raw[:256])

with tf.GradientTape() as tape:
    pred = m_diag(X_batch, training=True)
    loss = loss_fn(y_batch, pred)

grads = tape.gradient(loss, m_diag.trainable_variables)

print("Flusso dei gradienti per strato:")
print(f"{'Strato':15} {'Shape':20} {'media |grad|':>14} {'max |grad|':>12}")
print('-' * 65)
for var, grad in zip(m_diag.trainable_variables, grads):
    if grad is not None and 'kernel' in var.name:
        g = np.abs(grad.numpy())
        print(f"{var.name[:14]:15} {str(var.shape):20} {g.mean():>14.6f} {g.max():>12.6f}")

# Visualizzare
grad_means = []
for var, grad in zip(m_diag.trainable_variables, grads):
    if grad is not None and 'kernel' in var.name:
        grad_means.append((var.name.split('/')[0], np.abs(grad.numpy()).mean()))

nombres, medias = zip(*grad_means)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(nombres, medias, color='steelblue')
ax.set_ylabel('Media |gradiente|'); ax.set_title('Flusso dei gradienti (prima dell\'addestramento)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

In [ ]:
# Distribuzione dei pesi prima e dopo l'addestramento
m_before = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

pesos_antes = m_before.layers[0].get_weights()[0].ravel()

m_before.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m_before.fit(X_tr, y_tr_raw, epochs=5, batch_size=256, verbose=0)

pesos_despues = m_before.layers[0].get_weights()[0].ravel()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pesos_antes,   bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Pesi prima (media={pesos_antes.mean():.4f}, std={pesos_antes.std():.4f})')
axes[0].set_xlabel('Valore del peso')

axes[1].hist(pesos_despues, bins=60, color='coral',     edgecolor='white', alpha=0.8)
axes[1].set_title(f'Pesi dopo (media={pesos_despues.mean():.4f}, std={pesos_despues.std():.4f})')
axes[1].set_xlabel('Valore del peso')

plt.suptitle('Distribuzione dei pesi: strato Dense(128) prima e dopo l\'addestramento', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Rilevare 'neuroni morti' (Dead ReLU)
# Un neurone ReLU 'muore' se produce sempre 0 → non contribuisce all'apprendimento

tf.random.set_seed(42)
m_relu = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu', name='capa_relu'),
    layers.Dense(10, activation='softmax'),
])
m_relu.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m_relu.fit(X_tr, y_tr_raw, epochs=10, batch_size=256, verbose=0)

# Calcolare le attivazioni su un batch grande
m_relu_interm = keras.Model(inputs=m_relu.input,
                             outputs=m_relu.get_layer('capa_relu').output)
activaciones  = m_relu_interm.predict(X_te, batch_size=512, verbose=0)

n_neuronas_total  = activaciones.shape[1]
n_muestras_test   = activaciones.shape[0]

# Neurone 'morto' se si attiva in <1% dei campioni di test
tasa_activacion = (activaciones > 0).mean(axis=0)
n_muertas       = (tasa_activacion < 0.01).sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(tasa_activacion, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(0.01, color='red', ls='--', label='Soglia morto (1%)')
axes[0].set_xlabel('Frazione di campioni in cui si attiva'); axes[0].set_ylabel('Neuroni')
axes[0].set_title(f'Distribuzione dell\'attivazione — {n_muertas}/{n_neuronas_total} morti')
axes[0].legend()

# Attivazioni medie per neurone
axes[1].bar(range(n_neuronas_total), np.sort(tasa_activacion),
            color=['red' if t < 0.01 else 'steelblue' for t in np.sort(tasa_activacion)])
axes[1].axhline(0.01, color='red', ls='--', lw=1)
axes[1].set_xlabel('Neurone (ordinato)'); axes[1].set_ylabel('Tasso di attivazione')
axes[1].set_title('Tasso di attivazione per neurone (rosso = morto)')

plt.tight_layout(); plt.show()
print(f"Neuroni morti: {n_muertas}/{n_neuronas_total} ({n_muertas/n_neuronas_total:.1%})")

---
## 5. Flusso di lavoro professionale

In [ ]:
# Flusso di lavoro completo per un progetto di DL
from sklearn.metrics import classification_report

print("""
╔═══════════════════════════════════════════════════════════╗
║       FLUSSO DI LAVORO PROFESSIONALE NEL DEEP LEARNING    ║
╠═══════════════════════════════════════════════════════════╣
║ 1. Preparare i dati       → normalizzare, split, augment  ║
║ 2. Baseline semplice      → Dense piccolo, poche epoche   ║
║ 3. Aumentare complessità  → più strati, più neuroni        ║
║ 4. Regolarizzare          → Dropout, BN, L2               ║
║ 5. Affinare lr e schedule → CosineDecay, Warmup           ║
║ 6. Monitorare             → TensorBoard, callback          ║
║ 7. Diagnosticare          → curve, gradienti, pesi        ║
║ 8. Valutare sul test      → solo alla fine, una sola volta║
║ 9. Salvare e documentare  → .keras + JSON di config        ║
╚═══════════════════════════════════════════════════════════╝
""")

# Esempio completo con tutte le buone pratiche
def entrenar_modelo_profesional(X_tr, y_tr, X_te, y_te,
                                 nombre='modelo', log_dir='logs'):
    """Pipeline professionale di addestramento."""
    import json

    # 1. Architettura
    tf.random.set_seed(42)
    modelo = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(10, activation='softmax'),
    ], name=nombre)

    # 2. LR schedule
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001, decay_steps=len(X_tr)//256*20
    )

    # 3. Compilare
    modelo.compile(
        optimizer=keras.optimizers.Adam(lr_schedule),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # 4. Callback
    ts = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
    cbs = [
        keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(f'{nombre}_best.keras', save_best_only=True, verbose=0),
        keras.callbacks.TensorBoard(f'{log_dir}/{nombre}/{ts}', histogram_freq=1),
    ]

    # 5. Addestrare
    hist = modelo.fit(
        X_tr, y_tr,
        epochs=30, batch_size=256,
        validation_split=0.1,
        callbacks=cbs,
        verbose=0
    )

    # 6. Valutare sul test
    loss_te, acc_te = modelo.evaluate(X_te, y_te, verbose=0)
    n_epocas = len(hist.history['loss'])

    # 7. Salvare i metadati
    config = {
        'nombre':         nombre,
        'timestamp':      ts,
        'n_epocas':       n_epocas,
        'val_acc_final':  hist.history['val_accuracy'][-1],
        'test_loss':      float(loss_te),
        'test_accuracy':  float(acc_te),
        'parametros':     modelo.count_params(),
    }
    with open(f'{nombre}_config.json', 'w') as f:
        json.dump(config, f, indent=2)

    print(f"\n[{nombre}]")
    print(f"  Epoche: {n_epocas}  Parametri: {modelo.count_params():,}")
    print(f"  Val accuracy: {hist.history['val_accuracy'][-1]:.4f}")
    print(f"  Test accuracy: {acc_te:.4f}")
    return modelo, hist, config

modelo_final, hist_final, config = entrenar_modelo_profesional(
    X_tr, y_tr_raw, X_te, y_te_raw, nombre='mnist_profesional'
)

print("\nConfig salvata in mnist_profesional_config.json")

---
## Pratica

### Esercizio 1 — TensorBoard comparativo

**Consegna:** TensorBoard permette di confrontare più esperimenti nella stessa schermata.

1. Addestra 3 modelli con architetture diverse su Fashion MNIST
2. Salva ciascuno in una sottocartella diversa di `logs/experimentos/`
3. Lancia TensorBoard puntando a `logs/experimentos/`
4. Confronta le curve di loss nello stesso grafico

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import datetime

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

configs = [
    ('piccolo_32',   [32]),
    ('medio_128',    [128, 64]),
    ('grande_256',   [256, 128, 64]),
]

for nombre, capas_h in configs:
    tf.random.set_seed(42)
    capas = [layers.Input(shape=(784,))]
    for n in capas_h:
        capas.extend([layers.Dense(n, activation='relu'), layers.Dropout(0.3)])
    capas.append(layers.Dense(10, activation='softmax'))
    m = keras.Sequential(capas, name=nombre)
    m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

    ts = datetime.datetime.now().strftime('%H%M%S')
    tb = keras.callbacks.TensorBoard(f'logs/experimentos/{nombre}_{ts}', histogram_freq=0)
    m.fit(X_tr, y_tr, epochs=10, batch_size=256, validation_split=0.1,
          callbacks=[tb], verbose=0)
    _, acc = m.evaluate(X_te, y_te, verbose=0)
    print(f"{nombre}: {m.count_params():,} params, acc={acc:.4f}")

print("\nLancia: tensorboard --logdir logs/experimentos")
```
</details>

---
### Esercizio 2 — WarmupCosineDecay da zero

**Consegna:** Implementa uno schedule di lr personalizzato come sottoclasse di `keras.optimizers.schedules.LearningRateSchedule`:

1. Primi `warmup_steps` passi: lr sale linearmente da 0 a `peak_lr`
2. Dopo: cosine decay fino a `min_lr`
3. Usalo su un modello MNIST e visualizza la curva del lr durante l'addestramento

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps, total_steps, min_lr=1e-6):
        super().__init__()
        self.peak_lr      = peak_lr
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.min_lr       = min_lr

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / self.warmup_steps)
        decay_progress = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        cosine_lr = self.min_lr + 0.5*(self.peak_lr - self.min_lr)*(1 + tf.cos(np.pi * decay_progress))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {'peak_lr': self.peak_lr, 'warmup_steps': int(self.warmup_steps.numpy()),
                'total_steps': int(self.total_steps.numpy()), 'min_lr': self.min_lr}

# Visualizzare lo schedule
(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

N_EPOCAS = 15; BATCH = 256
pasos_por_epoca = len(X_tr) // BATCH
total_steps  = N_EPOCAS * pasos_por_epoca
warmup_steps = 2 * pasos_por_epoca  # 2 epoche di warmup

schedule = WarmupCosineDecay(peak_lr=0.002, warmup_steps=warmup_steps, total_steps=total_steps)

pasos = np.arange(total_steps)
lrs   = [float(schedule(p)) for p in pasos]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(pasos, lrs, 'steelblue', lw=2)
ax.axvline(warmup_steps, color='red', ls='--', label=f'Warmup ({warmup_steps} passi)')
ax.set_xlabel('Passo'); ax.set_ylabel('Learning rate')
ax.set_title('WarmupCosineDecay')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Usare nel modello
tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
m.compile(keras.optimizers.Adam(schedule), 'sparse_categorical_crossentropy', ['accuracy'])
m.fit(X_tr, y_tr, epochs=N_EPOCAS, batch_size=BATCH, validation_split=0.1, verbose=0)
_, acc = m.evaluate(X_te, y_te, verbose=0)
print(f"Test accuracy: {acc:.4f}")
```
</details>

---
### Esercizio 3 — Callback di stress test

**Consegna:** Crea un callback che, ogni 5 epoche, valuti il modello su un insieme di 'campioni difficili' (quelli con confidenza più bassa) e li registri:

1. `on_epoch_end`: se `epoch % 5 == 0`, valuta su X_te, individua i 20 campioni con minore confidenza
2. Salva un log di quanti di quei 20 sono corretti
3. Alla fine dell'addestramento, plotta come evolve quel numero

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

class PruebaEstres(keras.callbacks.Callback):
    def __init__(self, X_te, y_te, n_dificiles=20, frecuencia=5):
        super().__init__()
        self.X_te        = X_te
        self.y_te        = y_te
        self.n           = n_dificiles
        self.frecuencia  = frecuencia
        self.correctas   = []  # per controllo
        self.epocas_log  = []

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.frecuencia != 0:
            return
        probas    = self.model.predict(self.X_te, verbose=0)
        max_proba = probas.max(axis=1)  # confidenza massima per campione
        idx_dif   = np.argsort(max_proba)[:self.n]  # i n con confidenza minore
        y_pred    = probas[idx_dif].argmax(axis=1)
        n_ok      = (y_pred == self.y_te[idx_dif]).sum()
        self.correctas.append(n_ok)
        self.epocas_log.append(epoch + 1)
        print(f"  [Stress epoca {epoch+1}] {n_ok}/{self.n} corrette tra le più difficili")

    def on_train_end(self, logs=None):
        if not self.epocas_log:
            return
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(self.epocas_log, self.correctas, 'o-', color='steelblue', lw=2)
        ax.set_xlabel('Epoca'); ax.set_ylabel(f'Corrette delle {self.n} più difficili')
        ax.set_title('Evoluzione delle prestazioni sui campioni difficili')
        ax.set_ylim(0, self.n + 1); ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m.fit(X_tr, y_tr, epochs=25, batch_size=256, validation_split=0.1,
      callbacks=[PruebaEstres(X_te, y_te)], verbose=0)
```
</details>

---
### Esercizio 4 — Analisi degli errori per classe

**Consegna:** Non tutti gli errori sono uguali. Analizza gli errori del modello MNIST per classe:

1. Calcola precision, recall e F1 per cifra
2. Identifica le 5 coppie di cifre che si confondono di più
3. Visualizza i 3 campioni più errati di ogni coppia confusa
4. Quali pattern visivi spiegano le confusioni?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

y_pred = modelo_final.predict(X_te, verbose=0).argmax(axis=1)

print("Report per cifra:")
print(classification_report(y_te_raw, y_pred, target_names=[str(i) for i in range(10)]))

# Coppie più confuse (esclusa la diagonale)
cm = confusion_matrix(y_te_raw, y_pred)
np.fill_diagonal(cm, 0)

# Top 5 coppie
top5 = []
cm_copy = cm.copy()
for _ in range(5):
    idx = np.unravel_index(cm_copy.argmax(), cm_copy.shape)
    top5.append((idx[0], idx[1], cm_copy[idx]))
    cm_copy[idx] = 0

print("\nTop 5 confusioni:")
for real, pred, n in top5:
    print(f"  {real} → {pred}: {n} volte")

# Visualizzare i campioni confusi
fig, axes = plt.subplots(len(top5), 3, figsize=(8, len(top5)*2.5))
for row, (real, pred, n) in enumerate(top5):
    # Campioni dove reale=real e predetto=pred
    mask = (y_te_raw == real) & (y_pred == pred)
    idx_err = np.where(mask)[0][:3]
    for col, i in enumerate(idx_err):
        ax = axes[row, col]
        ax.imshow(X_te_raw[i], cmap='gray')
        if col == 0:
            ax.set_ylabel(f'{real}→{pred} ({n})', fontsize=9, rotation=0, labelpad=40, va='center')
        ax.set_title(f'Reale:{real} Pred:{pred}', fontsize=8, color='red')
        ax.axis('off')
plt.suptitle('Campioni delle 5 coppie più confuse', fontsize=13)
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 5 (sfida) — Esperimento automatizzato

**Consegna:** Costruisci un sistema di esperimenti che addestri diverse configurazioni, salvi i risultati e selezioni automaticamente la migliore:

1. Definisci una lista di 4 configurazioni (batch_size, dropout, lr)
2. Addestra ciascuna su Fashion MNIST con 10 epoche
3. Salva i risultati in un CSV (`experimentos.csv`)
4. Carica il CSV con pandas, ordina per val_acc e riaddestra la configurazione migliore per 30 epoche
5. Plotta la tabella dei risultati

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

configuraciones = [
    {'batch': 64,  'dropout': 0.2, 'lr': 0.001},
    {'batch': 128, 'dropout': 0.3, 'lr': 0.001},
    {'batch': 256, 'dropout': 0.4, 'lr': 0.0005},
    {'batch': 128, 'dropout': 0.3, 'lr': 0.003},
]

resultados = []
for i, cfg in enumerate(configuraciones):
    tf.random.set_seed(42)
    m = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(cfg['dropout']),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])
    m.compile(keras.optimizers.Adam(cfg['lr']), 'sparse_categorical_crossentropy', ['accuracy'])
    h = m.fit(X_tr, y_tr, epochs=10, batch_size=cfg['batch'],
              validation_split=0.1, verbose=0)
    _, acc_te = m.evaluate(X_te, y_te, verbose=0)
    res = {
        'exp': i, 'batch': cfg['batch'], 'dropout': cfg['dropout'], 'lr': cfg['lr'],
        'val_acc': h.history['val_accuracy'][-1], 'test_acc': acc_te
    }
    resultados.append(res)
    print(f"Exp {i}: val_acc={res['val_acc']:.4f}, test_acc={acc_te:.4f}")

df = pd.DataFrame(resultados).sort_values('val_acc', ascending=False)
df.to_csv('experimentos.csv', index=False)
print("\nRisultati salvati in experimentos.csv")
print(df.to_string(index=False))

mejor = df.iloc[0]
print(f"\nMigliore: batch={int(mejor.batch)}, dropout={mejor.dropout}, lr={mejor.lr}")

# Riaddestrare la migliore per 30 epoche
tf.random.set_seed(42)
m_final = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(mejor.dropout),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
m_final.compile(keras.optimizers.Adam(mejor.lr), 'sparse_categorical_crossentropy', ['accuracy'])
m_final.fit(X_tr, y_tr, epochs=30, batch_size=int(mejor.batch), validation_split=0.1, verbose=0)
_, acc_f = m_final.evaluate(X_te, y_te, verbose=0)
print(f"Configurazione migliore, 30 epoche → test_acc={acc_f:.4f}")
```
</details>

---
## Riepilogo

```python
# TensorBoard
tb = keras.callbacks.TensorBoard(
    log_dir='logs/experimento',
    histogram_freq=1,    # pesi e gradienti
    write_graph=True,
)
# Terminale: tensorboard --logdir logs/

# LR Schedule
lr = keras.optimizers.schedules.CosineDecay(initial_lr=0.001, decay_steps=N)
lr = keras.optimizers.schedules.ExponentialDecay(initial_lr=0.01, decay_steps=1000, decay_rate=0.9)
keras.callbacks.LearningRateScheduler(lambda ep: 0.001 * 0.9**ep)
keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

# Callback personalizzato
class MiCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # logs contiene: loss, accuracy, val_loss, val_accuracy
        pass
    def on_train_end(self, logs=None):
        pass

# Diagnosi
with tf.GradientTape() as tape:
    pred = model(X_batch, training=True)
    loss = loss_fn(y_batch, pred)
grads = tape.gradient(loss, model.trainable_variables)

# Flusso professionale: includere sempre
EarlyStopping + ModelCheckpoint + TensorBoard + LR schedule
```

## Cosa viene nella prossima lezione?
Nella **Lezione 10** (Progetto finale) costruiremo un classificatore di immagini completo con TensorFlow/Keras, applicando tutto quanto appreso nel modulo.